<a href="https://colab.research.google.com/github/NathanAndrewsBYU/Stylometry-Federalist-Papers/blob/main/notebooks/gpt2_federalist_stylometry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT-2 Authorship Attribution of the Federalist Papers

This notebook validates the perplexity-based authorship attribution method (following the 'Authorial Language Models' approach of Huang, Murakami & Grieve, 2025, PLOS ONE) on the Federalist Papers.

**What this notebook does:**
1. Download the full Federalist Papers text and split it into 85 individual files, sorted by known author.
2. Fine-tune a separate GPT-2 model on Hamilton's undisputed papers and on Madison's undisputed papers.
3. Run **leave-one-out validation**: hold out one known paper at a time, retrain on the rest, and check whether the method correctly identifies its true author. This gives you a measurable accuracy rate before trusting the method on anything contested.
4. Score the 12 historically disputed papers under both finished models and compare against the established scholarly consensus (nearly all statistical studies attribute all 12 to Madison).

**How to use this in Colab:** upload this file at colab.research.google.com, set Runtime > Change runtime type > GPU, then run cells top to bottom.

## 1. Install dependencies

In [ ]:
!pip install -q transformers torch datasets accelerate requests

## 2. Imports and GPU check

In [ ]:
import os
import re
import glob
import random
import requests
import torch
from transformers import (
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from torch.utils.data import Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type != "cuda":
    print("WARNING: no GPU detected. Go to Runtime > Change runtime type > GPU.")

Using device: cuda


## 3. Load base GPT-2

In [ ]:
MODEL_NAME = "gpt2"  # options: gpt2, gpt2-medium, gpt2-large, gpt2-xl

tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

base_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)
print(f"Loaded {MODEL_NAME}: {base_model.num_parameters():,} parameters")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loaded gpt2: 124,439,808 parameters


## 4. Download and split the Federalist Papers

This downloads the full text from Project Gutenberg and splits it into 85 individual files by detecting each paper's "FEDERALIST No. X" header. It then sorts each file into the correct author folder using the well-established scholarly attribution list (Adair 1944; consensus reflected in Mosteller & Wallace 1963):

- **Hamilton (51 papers, solely attributed):** 1, 6-9, 11-13, 15-17, 21-36, 59-61, 65-85
- **Madison (14 papers, solely attributed):** 10, 14, 37-48
- **Jay (5 papers):** 2, 3, 4, 5, 64
- **Joint Hamilton & Madison (3 papers, excluded from training):** 18, 19, 20
- **Disputed (12 papers, held out as the test set):** 49-58, 62, 63. Nearly all quantitative stylometric studies attribute these to Madison, but this remains formally contested, which is why it's a good test case.

In [ ]:
GUTENBERG_URL = "https://www.gutenberg.org/cache/epub/1404/pg1404.txt"

response = requests.get(GUTENBERG_URL)
response.raise_for_status()
raw_text = response.text

with open("federalist_raw.txt", "w", encoding="utf-8") as f:
    f.write(raw_text)

print(f"Downloaded {len(raw_text):,} characters.")
print("First 500 characters (sanity check):\n")
print(raw_text[:500])

Downloaded 1,186,743 characters.
First 500 characters (sanity check):

The Project Gutenberg eBook of The Federalist Papers
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using thi


In [ ]:
# Strip Gutenberg's boilerplate header/footer before splitting
start_marker = "*** START OF"
end_marker = "*** END OF"

start_idx = raw_text.find(start_marker)
end_idx = raw_text.find(end_marker)
if start_idx != -1 and end_idx != -1:
    body_start = raw_text.find("\n", start_idx) + 1
    body_text = raw_text[body_start:end_idx]
else:
    print("WARNING: could not find Gutenberg markers, using full text (check for boilerplate contamination).")
    body_text = raw_text

print(f"Body text length: {len(body_text):,} characters")

Body text length: 1,166,980 characters


In [ ]:
# Split into individual papers by header. Adjust this pattern if your downloaded
# edition formats headers differently (check federalist_raw.txt manually if needed).
HEADER_PATTERN = re.compile(r"FEDERALIST\.?\s+No\.?\s+(\d+)", re.IGNORECASE)

matches = list(HEADER_PATTERN.finditer(body_text))
print(f"Found {len(matches)} header matches (expect at least 85).")

papers = {}
for i, match in enumerate(matches):
    number = int(match.group(1))
    section_start = match.end()
    section_end = matches[i + 1].start() if i + 1 < len(matches) else len(body_text)
    text = body_text[section_start:section_end].strip()
    # Keep the longest version found for a given number, in case of duplicate
    # header mentions (e.g. table of contents entries also match the pattern).
    if number not in papers or len(text) > len(papers[number]):
        papers[number] = text

print(f"Distinct paper numbers recovered: {len(papers)}")
missing = sorted(set(range(1, 86)) - set(papers.keys()))
if missing:
    print(f"WARNING: missing papers {missing} — inspect federalist_raw.txt and adjust HEADER_PATTERN.")

Found 85 header matches (expect at least 85).
Distinct paper numbers recovered: 85


In [ ]:
# Sort each paper number into the correct author folder
HAMILTON_SOLO = set([1] + list(range(6, 10)) + list(range(11, 14)) + list(range(15, 18)) +
                     list(range(21, 37)) + list(range(59, 62)) + list(range(65, 86)))
MADISON_SOLO = set([10, 14] + list(range(37, 49)))
JAY = set([2, 3, 4, 5, 64])
JOINT = set([18, 19, 20])
DISPUTED = set(list(range(49, 59)) + [62, 63])

assert len(HAMILTON_SOLO) == 51, f"Hamilton count wrong: {len(HAMILTON_SOLO)}"
assert len(MADISON_SOLO) == 14, f"Madison count wrong: {len(MADISON_SOLO)}"
assert len(JAY) == 5
assert len(JOINT) == 3
assert len(DISPUTED) == 12

CORPUS_ROOT = "corpus/federalist"
for folder in ["hamilton", "madison", "jay", "joint", "disputed"]:
    os.makedirs(os.path.join(CORPUS_ROOT, folder), exist_ok=True)

def folder_for(number):
    if number in HAMILTON_SOLO:
        return "hamilton"
    if number in MADISON_SOLO:
        return "madison"
    if number in JAY:
        return "jay"
    if number in JOINT:
        return "joint"
    if number in DISPUTED:
        return "disputed"
    return None

written_count = 0
for number, text in papers.items():
    folder = folder_for(number)
    if folder is None:
        print(f"WARNING: paper {number} not in any known category, skipping.")
        continue
    out_path = os.path.join(CORPUS_ROOT, folder, f"federalist_{number:02d}.txt")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(text)
    written_count += 1

print(f"Wrote {written_count} paper files into {CORPUS_ROOT}/")
for folder in ["hamilton", "madison", "jay", "joint", "disputed"]:
    n = len(glob.glob(os.path.join(CORPUS_ROOT, folder, "*.txt")))
    print(f"  {folder}: {n} files")

Wrote 85 paper files into corpus/federalist/
  hamilton: 51 files
  madison: 14 files
  jay: 5 files
  joint: 3 files
  disputed: 12 files


## 5. Text cleaning helper

In [ ]:
def clean_text(raw):
    text = raw
    text = re.sub(r"\[.*?\]", "", text)          # editorial brackets
    text = re.sub(r"PUBLIUS\.?\s*$", "", text.strip())  # trailing signature
    text = re.sub(r"\s+", " ", text)              # collapse whitespace
    return text.strip()

def load_and_clean(file_list):
    chunks = []
    for path in file_list:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            chunks.append(clean_text(f.read()))
    return chunks

def list_txt_files(subfolder):
    return sorted(glob.glob(os.path.join(CORPUS_ROOT, subfolder, "*.txt")))

## 6. Perplexity function

In [ ]:
def compute_perplexity(model, tok, text, stride=512, max_length=1024):
    """Sliding-window perplexity for texts longer than the model's context window."""
    encodings = tok(text, return_tensors="pt")
    input_ids = encodings.input_ids.to(device)
    seq_len = input_ids.size(1)

    nlls = []
    prev_end = 0
    for begin in range(0, seq_len, stride):
        end = min(begin + max_length, seq_len)
        trg_len = end - prev_end
        ids = input_ids[:, begin:end]
        target_ids = ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            outputs = model(ids, labels=target_ids)
            neg_log_likelihood = outputs.loss * trg_len

        nlls.append(neg_log_likelihood)
        prev_end = end
        if end == seq_len:
            break

    ppl = torch.exp(torch.stack(nlls).sum() / end)
    return ppl.item()

## 7. Fine-tuning function

In [ ]:
class SimpleTextDataset(Dataset):
    """Replacement for transformers' removed TextDataset class: tokenizes a text
    file and chunks it into fixed-length blocks for language model training."""
    def __init__(self, tokenizer, file_path, block_size=256):
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()
        token_ids = tokenizer.encode(text)
        self.examples = [
            token_ids[i:i + block_size]
            for i in range(0, len(token_ids) - block_size + 1, block_size)
        ]
        if not self.examples:
            # corpus shorter than one block: use whatever we have, padded by the collator
            self.examples = [token_ids]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return torch.tensor(self.examples[idx], dtype=torch.long)

def prepare_training_file(file_list, out_path):
    texts = load_and_clean(file_list)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n\n".join(texts))
    return out_path

def fine_tune_on_files(name, file_list, output_dir, epochs=100, block_size=256):
    """epochs=100 mirrors the hyperparameter reported in Huang, Murakami & Grieve's
    ALMs paper as a starting point — tune down if you see overfitting on small corpora."""
    train_file = prepare_training_file(file_list, f"{name}_train.txt")

    dataset = SimpleTextDataset(tokenizer=tokenizer, file_path=train_file, block_size=block_size)
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=2,
        save_strategy="no",
        logging_steps=50,
        report_to=[],
    )

    trainer = Trainer(model=model, args=training_args, data_collator=data_collator, train_dataset=dataset)
    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    return output_dir

def load_finetuned_model(model_dir):
    model = GPT2LMHeadModel.from_pretrained(model_dir).to(device)
    tok = GPT2TokenizerFast.from_pretrained(model_dir)
    return model, tok

## 8. Leave-one-out validation on known-authorship papers

In [ ]:
def leave_one_out_validation(hamilton_files, madison_files, sample_size=None, epochs=100):
    """
    Returns a list of dicts: {file, true_author, predicted_author, ppl_hamilton, ppl_madison, correct}
    Set sample_size to an integer to test on a random subset first (faster sanity check).
    """
    results = []

    all_papers = [("hamilton", f) for f in hamilton_files] + [("madison", f) for f in madison_files]
    if sample_size:
        all_papers = random.sample(all_papers, min(sample_size, len(all_papers)))

    for true_author, held_out_file in all_papers:
        remaining_hamilton = [f for f in hamilton_files if f != held_out_file]
        remaining_madison = [f for f in madison_files if f != held_out_file]

        ham_dir = fine_tune_on_files("loo_hamilton", remaining_hamilton, "models/loo_hamilton", epochs=epochs)
        mad_dir = fine_tune_on_files("loo_madison", remaining_madison, "models/loo_madison", epochs=epochs)

        ham_model, ham_tok = load_finetuned_model(ham_dir)
        mad_model, mad_tok = load_finetuned_model(mad_dir)

        held_out_text = load_and_clean([held_out_file])[0]
        ppl_hamilton = compute_perplexity(ham_model, ham_tok, held_out_text)
        ppl_madison = compute_perplexity(mad_model, mad_tok, held_out_text)

        predicted_author = "hamilton" if ppl_hamilton < ppl_madison else "madison"
        correct = predicted_author == true_author

        results.append({
            "file": os.path.basename(held_out_file),
            "true_author": true_author,
            "predicted_author": predicted_author,
            "ppl_hamilton": ppl_hamilton,
            "ppl_madison": ppl_madison,
            "correct": correct,
        })
        print(f"{os.path.basename(held_out_file)} | true={true_author} pred={predicted_author} "
              f"(H:{ppl_hamilton:.1f} M:{ppl_madison:.1f}) {'CORRECT' if correct else 'WRONG'}")

    accuracy = sum(r["correct"] for r in results) / len(results)
    print(f"\nOverall leave-one-out accuracy: {accuracy:.1%} ({sum(r['correct'] for r in results)}/{len(results)})")
    return results

 #Example usage — start small to sanity-check before running the full set overnight:
hamilton_files = list_txt_files("hamilton")
madison_files = list_txt_files("madison")
results = leave_one_out_validation(hamilton_files, madison_files, sample_size=65, epochs=10)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (132711 > 1024). Running this sequence through the model will result in indexing errors


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,3.512356
100,3.417878
150,3.377775
200,3.345045
250,3.346542
300,3.115342
350,3.046309
400,3.065350
450,3.037044
500,3.026486


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2438 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2438 > 1024). Running this sequence through the model will result in indexing errors


federalist_16.txt | true=hamilton pred=hamilton (H:29.1 M:41.0) CORRECT


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Step,Training Loss
50,3.526127
100,3.453286
150,3.399473
200,3.340621
250,3.297894
300,3.106010
350,3.031928
400,3.067983
450,3.015381
500,3.032805


In [ ]:
import csv

with open("leave_one_out_n65.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)

print(f"Saved {len(results)} results to leave_one_out_n65.csv")

NameError: name 'results' is not defined

## 9. Score the 12 disputed papers

Train final Hamilton and Madison models on ALL their solely-attributed papers (no held-out this time), then score each disputed paper under both.

In [ ]:
def score_disputed_papers(hamilton_files, madison_files, disputed_files, epochs=100):
    ham_dir = fine_tune_on_files("final_hamilton", hamilton_files, "models/final_hamilton", epochs=epochs)
    mad_dir = fine_tune_on_files("final_madison", madison_files, "models/final_madison", epochs=epochs)

    ham_model, ham_tok = load_finetuned_model(ham_dir)
    mad_model, mad_tok = load_finetuned_model(mad_dir)

    results = []
    for path in disputed_files:
        text = load_and_clean([path])[0]
        ppl_h = compute_perplexity(ham_model, ham_tok, text)
        ppl_m = compute_perplexity(mad_model, mad_tok, text)
        verdict = "Hamilton" if ppl_h < ppl_m else "Madison"
        results.append({"file": os.path.basename(path), "ppl_hamilton": ppl_h, "ppl_madison": ppl_m, "verdict": verdict})
        print(f"{os.path.basename(path)}: H={ppl_h:.1f}  M={ppl_m:.1f}  -> {verdict}")

    madison_count = sum(1 for r in results if r["verdict"] == "Madison")
    print(f"\n{madison_count}/{len(results)} disputed papers attributed to Madison "
          f"(scholarly consensus attributes all 12 to Madison).")
    return results

# Example usage:
# disputed_files = list_txt_files("disputed")
# disputed_results = score_disputed_papers(hamilton_files, madison_files, disputed_files)